## Load useful libraries

In [1]:
import os
import glob
import xml.etree.ElementTree as ET
import pandas as pd
import xmltodict
import numpy as np
import html
import pickle

## User settings

In [2]:
directory_data = '/home/emily/Downloads/pubmed'
url_root = 'https://pubmed.ncbi.nlm.nih.gov'

## Retrieve a list of XML files to process

In [3]:
list_files_xml = sorted(glob.glob(directory_data + '/*.xml.gz'))

## Define a wrapper function

In [4]:
def decompression_wrapper(filename, list_functions_to_use):
    os.system('gunzip ' + filename)

    dict_results = {}
    for function_to_use in list_functions_to_use:

        function_name = str(function_to_use).split('function ')[1].split(' ')[0]
        dict_results[function_name] = function_to_use(filename.replace('.gz', ''))
    
    os.system('gzip ' + filename.replace('.gz', ''))
    return dict_results

## Define functions that extract ONLY PubMed article titles and IDs

#### Method 1

In [5]:
def extract_all_article_ids_and_titles_from_XML_file_method_1(xml_file : str) -> pd.DataFrame:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    extracted_data = []

    for article in root.findall(".//PubmedArticle"):
        pmid_str = article.findtext(".//PMID").strip()  # kept a string for now

        title_element = article.find(".//ArticleTitle")
        title = ET.tostring(title_element, encoding="unicode").strip()
        title = title.replace('ArticleTitle>', '')[1:-2]
        title = html.unescape(title)
        
        if pmid_str != '' and title != '':
            extracted_data.append({'pmid_str' : pmid_str, 'title' : title})

    df = pd.DataFrame(extracted_data)
    return df

#### Method 2

In [6]:
def extract_all_article_ids_and_titles_from_XML_file_method_2(xml_file : str) -> pd.DataFrame:

    with open(xml_file, 'r', encoding='utf-8') as file:
        xml_data = file.read()
    dict_pm = xmltodict.parse(xml_data)
    
    extracted_data = []
    for item in dict_pm['PubmedArticleSet']['PubmedArticle']:
        article = item['MedlineCitation']['Article']
        pmid_str = item['MedlineCitation']['PMID']['#text']
        
        title = article['ArticleTitle']
        if isinstance(title, dict):

            # WTF?
            if '#text' in title:
                new_title = title['#text']
            elif 'i' in title:
                new_title = title['i']
            else:
                new_title = 'FAILED TITLE GRAB'

            title = new_title
        
        if pmid_str != '' and title != '':
            extracted_data.append({'pmid_str' : pmid_str, 'title' : title})

    df = pd.DataFrame(extracted_data)
    return df

## Run the two ID/title methods

In [7]:
#import pickle
#with open('temp.pickled', 'rb') as f:
#    list_dict_id_title = pickle.load(f)
#list_already_processed = [x['filename'] for x in list_dict_id_title]

In [8]:
#print(len(list_already_processed))

In [9]:
#print(len(list_files_xml))

In [10]:
## update
#list_files_xml = sorted(list(set(list_files_xml) - set(list_already_processed)))

In [11]:
#print(len(list_files_xml))

In [ ]:
list_dict_id_title = []
for filename in list_files_xml:

    #print(filename)
    
    dict_results = decompression_wrapper(
        filename, [
            extract_all_article_ids_and_titles_from_XML_file_method_1,
            extract_all_article_ids_and_titles_from_XML_file_method_2,
        ]
    )
    dict_results['filename'] = filename
    list_dict_id_title.append(dict_results)

## Save temp

In [ ]:
import pickle
with open('temp2.pickled', 'wb') as f:
    pickle.dump(list_dict_id_title, f)

In [ ]:
#[x['filename'] for x in list_dict_id_title][0:2]

In [ ]:
#import pickle
#with open('temp.pickled', 'rb') as f:
#    list_dict_id_title = pickle.load(f)
#list_already_processed = [x['filename'] for x in list_dict_id_title]
#list_already_processed[-2:]

In [ ]:
list_dict_id_title[0]

## Investigate the difference between the two methods

In [ ]:
dict_no_na = {}
for how in ['left', 'right']:
    dict_no_na[how] = {}
    for dict_id_title in list_dict_id_title:
        filename = dict_id_title['filename']
        list_keys = [x for x in list(dict_id_title.keys()) if not x in ['filename']]

        for i, key in enumerate(list_keys):
            if i == 0:
                df = dict_id_title[key]
                df = df.rename(columns = {'title' : 'title_' + str(i)})
            else:
                df_next = dict_id_title[key]
                df_next = df_next.rename(columns = {'title' : 'title_' + str(i)})
                df = pd.merge(df, df_next, on = 'pmid_str', how = how)


        no_na = len(df.index) == len(df.dropna().index)
        if not no_na:
            dict_no_na[how][filename] = no_na
    
    
    
    #for i in range(0, len(list_keys) - 1):
    #    list_no_errors.append(bool(np.min(np.int8((df['title_' + str(i)] == df['title_' + str(i + 1)]).values)) == 1))

    #print(filename, min([int(x) for x in list_no_errors]) == 1)

    #print()
    #print()
    #print(df[df['title_0'] != df['title_1']])

In [ ]:
dict_no_na

In [ ]:
# stopped here

In [ ]:
def extract_pubmed_metadata(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    extracted_data = []

    for article in root.findall(".//PubmedArticle"):
        # 1. PubMed ID (PMID)
        pmid = article.findtext(".//PMID")
        
        # 2. Article Title
        title = article.findtext(".//ArticleTitle")
        
        # 3. Keywords (Author-assigned)
        keywords = [kw.text for kw in article.findall(".//Keyword") if kw.text]
        
        # 4. MeSH Terms (Medical Subject Headings)
        mesh_descriptors = [
            mesh.get('UI') for mesh in article.findall(".//MeshHeading/DescriptorName")
            if mesh.text
        ]

        # temp
        if len(mesh_descriptors) > 0:
            
            extracted_data.append({
                'pmid' : pmid,
                'title' : title,
                'keywords' : keywords,
                'mesh_descriptors' : mesh_descriptors
            })

    return extracted_data

In [ ]:
#list_df = []
#for filename in pubmed_xml_list:

    #print(filename)
    
    #file_to_process = download_file('http://' + ftp_pubmed_root + '/' + remote_directory + '/' + filename, directory_download_to)
    #file_to_process_md5 = download_file('http://' + ftp_pubmed_root + '/' + remote_directory + '/' + filename + '.md5', directory_download_to)

    ## what happens when this fails?
    ##print(compute_file_md5(file_to_process) == read_md5_file(file_to_process_md5))
    
    #os.system('gunzip ' + file_to_process)
    #new_filename = '.'.join(file_to_process.split('.')[0:-1])
    #articles = extract_pubmed_metadata(new_filename)
    #os.system('gzip ' + new_filename) # pointless if we are going to simply delete the file immediately after use
    #list_df.append(pd.DataFrame(articles))
    #os.system('rm ' + file_to_process)
    #os.system('rm ' + file_to_process + '.md5')

In [ ]:
df = pd.concat(list_df)

In [ ]:
df

In [ ]:

import pprint as pp
import pandas as pd

In [ ]:
dict_pm_to_mesh = {}

In [ ]:
with open('/home/emily/Downloads/pubmed/pubmed26n0029.xml', 'r', encoding='utf-8') as file:
    xml_data = file.read()
dict_pm = xmltodict.parse(xml_data)

In [ ]:
count = 0
for item in dict_pm['PubmedArticleSet']['PubmedArticle']:
    
    if 'MeshHeadingList' in item['MedlineCitation'] or 'MeshHeading' in item['MedlineCitation']:
        list_mesh_heading = []
        
        # this probably won't work
        if 'MeshHeading' in item['MedlineCitation']:
            list_mesh_heading.extend(item['MedlineCitation']['MeshHeading'])
        
        # this works
        if 'MeshHeadingList' in item['MedlineCitation']:
            list_mesh_heading.extend(item['MedlineCitation']['MeshHeadingList']['MeshHeading'])



        article = item['MedlineCitation']['Article']
        #title = article['ArticleTitle']
        pmid = int(item['MedlineCitation']['PMID']['#text'])

        dict_pm_to_mesh[pmid] = {}
        #print(title)

        
        for mesh_item in list_mesh_heading:
            descriptor_ui = mesh_item['DescriptorName']['@UI']

            list_qualifiers = []
            if 'QualifierName' in mesh_item:
                qualifier_item_list = mesh_item['QualifierName']

                if not isinstance(qualifier_item_list, list):
                    qualifier_item_list = [qualifier_item_list]
                
                list_qualifiers.extend([x['@UI'] for x in qualifier_item_list])

            dict_pm_to_mesh[pmid][descriptor_ui] = list_qualifiers
            
#            print('\t', descriptor_ui, list_qualifiers)

#        print()



        count += 1

        if count > 3:  break

    
pp.pprint(dict_pm_to_mesh)

In [ ]:
list_pmid_has_d = []
for pmid in dict_pm_to_mesh.keys():
    for mesh_descriptor in dict_pm_to_mesh[pmid].keys():
        list_pmid_has_d.append({'pmid' : pmid, 'MeSH_descriptor' : mesh_descriptor})

df_pmid_has_d = pd.DataFrame(list_pmid_has_d).dropna().drop_duplicates().reset_index(drop = True)
df_pmid_has_d.head(3)

In [ ]:
list_d_to_q = []
for pmid in dict_pm_to_mesh.keys():
    for mesh_descriptor in dict_pm_to_mesh[pmid].keys():
        for q in dict_pm_to_mesh[pmid][mesh_descriptor]:
            list_d_to_q.append({'MeSH_descriptor' : mesh_descriptor, 'relationship' : pmid, 'MeSH_qualifier' : q})

df_d_to_q = pd.DataFrame(list_d_to_q).dropna().drop_duplicates().reset_index(drop = True)
df_d_to_q.head(11)